# ਪਾਠ 09 - ਮੈਟਾਕੋਗਨਿਸ਼ਨ ਡਿਜ਼ਾਈਨ ਪੈਟਰਨ


## ਸੈਟਅੱਪ

ਇਹ ਨੋਟਬੁੱਕ ਮਾਈਕ੍ਰੋਸਾਫਟ ਏਜੰਟ ਫਰੇਮਵਰਕ ਦੀ ਵਰਤੋਂ ਕਰਦੇ ਹੋਏ ਮੈਟਾਕੌਗਨੀਸ਼ਨ ਡਿਜ਼ਾਈਨ ਪੈਟਰਨ ਨੂੰ ਦਰਸਾਉਂਦਾ ਹੈ।

**ਪਹਿਲਾਂ ਦੀਆਂ ਲੋੜਾਂ:**
- ਵਾਤਾਵਰਣ ਚਰਾਂ ਰਾਹੀਂ ਸੰਰਚਿਤ ਅਜ਼ੂਰ ਓਪਨ AI ਡਿਪਲੌਇਮੈਂਟ
- ਅਜ਼ੂਰ CLI ਨਾਲ ਪ੍ਰਮਾਣਿਕ੍ਰਿਤ (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## ਮੈਟਾਕੌਗਨਿਸ਼ਨ ਕੀ ਹੈ?

ਮੈਟਾਕੌਗਨਿਸ਼ਨ ਦਾ ਮਤਲਬ ਹੈ **ਸੋਚ ਬਾਰੇ ਸੋਚਣਾ**। ਏਆਈ ਏਜੰਟਾਂ ਦੇ ਸੰਦਰਭ ਵਿੱਚ, ਇਸਦਾ ਅਰਥ ਹੈ ਅਜਿਹੇ ਏਜੰਟ ਬਣਾਉਣਾ ਜੋ ਸਕਦੇ ਹਨ:

- **ਆਪਣੇ ਨਤੀਜਿਆਂ ਅਤੇ ਤਰਕਸ਼ੀਲ ਪ੍ਰਕਿਰਿਆ 'ਤੇ ਖੁਦ ਦਾ ਆਇਨਾ ਦੇਖਣਾ**
- **ਗਲਤੀਆਂ ਦਾ ਪਤਾ ਲਗਾਉਣਾ** ਅਤੇ ਚੁਪਚਾਪ ਫੇਲ ਹੋਣ ਦੀ ਬਜਾਏ ਨਰਮਦਲੀ ਨਾਲ ਮੁੜ ਸਹੀ ਰਸਤਾ ਲੱਭਣਾ
- **ਅੰਕਣ ਕਰਨਾ** ਕਿ ਉਹਨਾਂ ਦੇ ਜਵਾਬ ਪੂਰੇ ਅਤੇ ਮਦਦਗਾਰ ਹਨ ਜਾਂ ਨਹੀਂ
- **ਆਪਣੀ ਰਣਨੀਤੀ ਬਦਲਣਾ** ਜਦੋਂ ਸ਼ੁਰੂਆਤੀ ਦਿਸ਼ਾ ਕੰਮ ਨਾ ਕਰੇ (ਜਿਵੇਂ ਕਿ ਬੈਕਅੱਪ ਸਿਸਟਮ 'ਤੇ ਜਾਉਣਾ)

ਇੱਕ ਮੈਟਾਕੌਗਨਿਟਿਵ ਏਜੰਟ ਸਿਰਫ਼ ਸਵਾਲਾਂ ਦੇ ਜਵਾਬ ਨਹੀਂ ਦਿੰਦਾ — ਇਹ ਆਪਣੀ ਕਾਰਗੁਜ਼ਾਰੀ ਦੀ ਨਿਗਰਾਨੀ ਕਰਦਾ ਹੈ ਅਤੇ ਤੁਰੰਤ ਅਨੁਕੂਲਤਾ ਕਰਦਾ ਹੈ।


## ਮੁੱਖ ਅਤੇ ਬੈਕਅੱਪ ਟੂਲ

ਇੱਕ ਆਮ ਮੈਟਾਕੋਗਨੀਸ਼ਨ ਪੈਟਰਨ **ਫਾਲਬੈਕ ਸਟ੍ਰੈਟਜੀ** ਹੈ। ਏਜੰਟ ਪਹਿਲਾਂ ਇੱਕ ਮੁੱਖ ਟੂਲ ਦੀ ਕੋਸ਼ਿਸ਼ ਕਰਦਾ ਹੈ; ਜੇ ਇਹ ਅਸਫਲ ਰਹਿੰਦਾ ਹੈ (ਜਿਵੇਂ ਕਿ, 404 ਐਰਰ), ਤਾਂ ਏਜੰਟ ਅਸਫਲਤਾ ਨੂੰ ਜਾਣ ਲੈਂਦਾ ਹੈ ਅਤੇ ਸਪਸ਼ਟ ਤੌਰ 'ਤੇ ਬੈਕਅੱਪ ਟੂਲ ਵੱਲ ਸਵਿੱਚ ਕਰ ਲੈਂਦਾ ਹੈ।

ਇਹ ਅਸਲ ਜ਼ਿੰਦਗੀ ਦੇ ਸਿਸਟਮਾਂ ਵਰਗਾ ਹੈ ਜਿੱਥੇ ਮੁੱਖ ਸੇਵਾਵਾਂ ਉਪਲਬਧ ਨਹੀਂ ਹੋ ਸਕਦੀਆਂ ਅਤੇ ਏਜੰਟ-ਆਪਣੇ ਆਪ ਮੁੱਦੇ ਦੀ ਪਹਚਾਣ ਕਰਕੇ ਬਦਲਵੀਂ ਰਾਹ ਚੁਣਦੇ ਹਨ।

ਹੇਠਾਂ ਅਸੀਂ ਦੋ ਫਲਾਈਟ ਲੁੱਕਅਪ ਟੂਲ ਸੰਜੋਦੇ ਹਾਂ:
- **ਮੁੱਖ** — ਪੈਰਿਸ, ਟੋਕਿਓ, ਅਤੇ ਬਾਰਸਿਲੋਨਾ ਨੂੰ ਕਵਰ ਕਰਦਾ ਹੈ
- **ਬੈਕਅੱਪ** — ਬਰਲਿਨ, ਸਿਡਨੀ, ਅਤੇ ਨਿਊ ਯਾਰ্ক ਸਿਟੀ ਨੂੰ ਕਵਰ ਕਰਦਾ ਹੈ


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## ਏਰਰ ਰਿਕਵਰੀ ਨਾਲ ਸਵੈ-ਪ੍ਰਤੀਬਿੰਬਿਤ ਏਜੰਟ

ਹੇਠਾਂ ਦਿੱਤਾ ਗਇਆ ਏਜੰਟ ਪ੍ਰਾਥਮਿਕ ਫਲਾਈਟ ਸਿਸਟਮ ਨੂੰ ਪਹਿਲਾਂ ਕੋਸ਼ਿਸ਼ ਕਰਨ, ਨਾਕਾਮੀਆਂ ਨੂੰ ਪਹਿਚਾਨਣ ਅਤੇ ਖੁੱਲ੍ਹੇ ਦਿਲ ਨਾਲ ਬੈਕਅਪ ਸਿਸਟਮ ’ਤੇ ਜਾਵੇਗਾ। ਹਰ ਜਵਾਬ ਤੋਂ ਬਾਅਦ ਇਹ ਸੰਖੇਪ ਵਿੱਚ ਖੁਦ ਦਾ ਮੁਲਾਂਕਣ ਕਰਦਾ ਹੈ ਕਿ ਉਸ ਨੇ ਯੂਜ਼ਰ ਦੇ ਸਵਾਲ ਦਾ ਪੂਰਾ ਜਵਾਬ ਦਿੱਤਾ ਹੈ ਜਾਂ ਨਹੀਂ।


In [ ]:
agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## ਸਵੈ-ਮੁਲਾਂਕਣ ਪੈਟਰਨ

ਮੈਟਾਕੋਗਨਿਸ਼ਨ ਦਾ ਇਕ ਹੋਰ ਪਹਲੂ ਹੈ **ਸਵੈ-ਮੁਲਾਂਕਣ**: ਇੱਕ ਵੱਖਰਾ ਏਜੰਟ (ਜਾਂ ਇੱਕੋ ਏਜੰਟ ਦੂਜੇ ਚੱਕਰ ਵਿੱਚ) ਜਵਾਬ ਨੂੰ ਪੂਰਨਤਾ, ਸਹੀਤਾ ਅਤੇ ਸਹਾਇਕਤਾ ਲਈ ਸਮੀਖਿਆ ਕਰਦਾ ਹੈ।

ਹੇਠਾਂ ਅਸੀਂ ਇੱਕ `ResponseEvaluator` ਏਜੰਟ ਬਣਾਉਂਦੇ ਹਾਂ ਜੋ ਯਾਤਰਾ ਏਜੰਟ ਦੇ ਜਵਾਬਾਂ ਨੂੰ ਤਿੰਨ ਆਯਾਮਾਂ 'ਤੇ ਪਾਇੰਟ ਕਰਦਾ ਹੈ।


In [ ]:
evaluation_agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## ਸੰਖੇਪ

ਇਸ ਪਾਠ ਵਿੱਚ ਤੁਸੀਂ ਸਿੱਖਿਆ ਕਿ ਕਿਵੇਂ ਮਾਈਕ੍ਰੋਸਾਫਟ ਏਜੰਟ ਫਰੇਮਵਰਕ ਦੀ ਵਰਤੋਂ ਕਰਕੇ **ਮੇਟਾਕੋਗਨਿਟਿਵ ਏਜੰਟਸ** ਬਣਾਏ ਜਾਂਦੇ ਹਨ:

- **ਸਵੈ-ਚਿੰਤਨ**: ਉਹ ਏਜੰਟ ਜੋ ਆਪਣੀ ਸੋਚ ਨੂੰ ਨਿਗਰਾਨੀ ਕਰਦੇ ਹਨ ਅਤੇ ਪਾਰਦਰਸ਼ੀ ਤਰੀਕੇ ਨਾਲ ਦੱਸਦੇ ਹਨ ਕਿ ਕੀ ਹੋਇਆ।
- **ਗਲਤੀ ਮੁਕਾਬਲੇ ਲਈ ਬੈਕਅਪ ਪੈਟਰਨ**: ਇੱਕ ਪ੍ਰਾਇਮਰੀ + ਬੈਕਅਪ ਟੂਲ ਪੈਟਰਨ ਜਿੱਥੇ ਏਜੰਟ ਅਸਫਲਤਾਵਾਂ (ਜਿਵੇਂ 404 ਗਲਤੀਆਂ) ਨੂੰ ਪਛਾਣਦਾ ਹੈ ਅਤੇ ਖੁਦਬਖੁਦ ਵਿਕਲਪਿਕ ਸਰੋਤ ਦੀ ਕੋਸ਼ਿਸ਼ ਕਰਦਾ ਹੈ।
- **ਸਵੈ-ਮੁੱਲਾਂਕਣ**: ਇੱਕ ਵੱਖਰਾ ਮੁੱਲਾਂਕਣ ਕਰਨ ਵਾਲਾ ਏਜੰਟ ਜੋ ਜਵਾਬਾਂ ਨੂੰ ਪੂਰਨਤਾ, ਸਹੀਤਾ ਅਤੇ ਸਹਾਇਤਾ ਲਈ ਅੰਕ ਦਿੰਦਾ ਹੈ।

ਇਹ ਪੈਟਰਨ ਏਜੰਟਸ ਨੂੰ ਹੋਰ ਮਜ਼ਬੂਤ, ਪਾਰਦਰਸ਼ੀ ਅਤੇ ਭਰੋਸੇਮੰਦ ਬਣਾਉਂਦੇ ਹਨ — ਪ੍ਰੋਡਕਸ਼ਨ ਤੈਨਾਤੀਆਂ ਲਈ ਬਹੁਤ ਹੀ ਜਰੂਰੀ ਗੁਣ।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ਅਸਵੀਕਾਰੋਪਣ**:
ਇਸ ਦਸਤਾਵੇਜ਼ ਦਾ ਅਨੁਵਾਦ ਏਆਈ ਅਨੁਵਾਦ ਸੇਵਾ [Co-op Translator](https://github.com/Azure/co-op-translator) ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਕੀਤਾ ਗਿਆ ਹੈ। ਜਦੋਂ ਕਿ ਅਸੀਂ ਸਹੀਤਾਵਾਂ ਲਈ ਯਤਨਸ਼ੀਲ ਹਾਂ, ਕਿਰਪਾ ਕਰਕੇ ਧਿਆਨ ਰੱਖੋ ਕਿ ਸਵੈਚਾਲਿਤ ਅਨੁਵਾਦਾਂ ਵਿੱਚ ਗਲਤੀਆਂ ਜਾਂ ਅਸਮੱਤਿਆਵਾਂ ਹੋ ਸਕਦੀਆਂ ਹਨ। ਮੂਲ ਦਸਤਾਵੇਜ਼ ਆਪਣੀ ਮੂਲ ਭਾਸ਼ਾ ਵਿੱਚ ਅਧਿਕਾਰਕ ਸਰੋਤ ਮੰਨਿਆ ਜਾਣਾ ਚਾਹੀਦਾ ਹੈ। ਜਰੂਰੀ ਜਾਣਕਾਰੀ ਲਈ, ਪੇਸ਼ੇਵਰ ਮਨੁੱਖੀ ਅਨੁਵਾਦ ਦੀ ਸਿਫ਼ਾਰਸ਼ ਕੀਤੀ ਜਾਂਦੀ ਹੈ। ਅਸੀਂ ਇਸ ਅਨੁਵਾਦ ਦੇ ਉਪਯੋਗ ਤੋਂ ਪੈਦਾ ਹੋਣ ਵਾਲੀਆਂ ਕਿਸੇ ਵੀ ਗਲਤਫਹਿਮੀਆਂ ਜਾਂ ਗਲਤ ਵਿਆਖਿਆਵਾਂ ਲਈ ਜਵਾਬਦੇਹ ਨਹੀਂ ਹਾਂ।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
